<a href="https://colab.research.google.com/github/dxda6216/ttron2excel/blob/main/ttron_data_file_to_excel_file.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import csv
import math
import numpy as np
from datetime import datetime, timedelta, timezone
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.ticker import MultipleLocator
import io
from google.colab import files
from scipy import signal # Added for sinc filter implementation

# Helper function for peak and trough detection
def find_peaks_and_troughs(series, distance_in_hours=None, time_interval=None):
    if series.isnull().all():
        return [], []

    data_to_analyze = series.dropna().values

    # Determine distance for peak finding, if not provided, use a default
    if distance_in_hours and time_interval:
        distance_points = max(1, int(distance_in_hours / time_interval))
    else:
        distance_points = 5 # Default distance in data points

    # Find peaks
    peaks, _ = signal.find_peaks(data_to_analyze, distance=distance_points)

    # Find troughs by inverting the signal
    troughs, _ = signal.find_peaks(-data_to_analyze, distance=distance_points)

    # Map back to original series indices (if NaNs were dropped)
    original_indices = series.dropna().index
    peaks_original_indices = original_indices[peaks]
    troughs_original_indices = original_indices[troughs]

    return peaks_original_indices, troughs_original_indices

#@title Converting Taylortron TRACES file (TRACES.nnn) to Excel file
#@markdown **This script works only with a specific format of data files (*TRACES.nnn* files) generated by the [Taylortron](https://doi.org/10.1080/09291018209359765) in the Carl Johnson Lab.**

#@markdown 1. Input the experiment number (avoid spaces and special characters).
#@markdown 2. Input the experiment title (this field can be blank).
#@markdown 3. Input the date on which the experiment started.
#@markdown 4. Select 'Sinc Filter' ([firwin](https://docs.scipy.org/docs/scipy/reference/generated/scipy.signal.firwin.html)) or '[Moving Average](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rolling.html)' for detrending.
#@markdown 5. If 'Sinc Filter' is chosen, adjust 'Sinc_Filter_Cutoff_Period_Hours' and 'Sinc_Filter_Order'.
#@markdown 6. If 'Moving Average' is chosen, adjust 'Window_size_for_trend_line_moving_average'.
#@markdown 7. **Runtime** -> **Restart and run all** (or press **Ctrl+M** and then press **Ctrl+F9**)
#@markdown 8. Wait until `Choose Files` or `Browse...` button appears below.
#@markdown 9. Click `Choose Files` or `Browse` button and select *TRACES.nnn* file in your computer.
#@markdown 10. Wait a while. two Excel files, one ZIP file, and one PDF file will be saved in "Downloads" folder in your computer.

#@markdown - The first Excel file will have multiple spreadsheets, conatining all the raw data, smoothed data, and detranded data.
#@markdown - The second Excel file will have a single spreadsheet, conatining only the raw time series data. The interval time will be indicated as a sheet name of the Excel spreadsheet. This file can be opened with data analysis programs such as [pyBOAT](https://github.com/tensionhead/pyBOAT).
#@markdown - The ZIP file will contain separate data files (.dat files) for each of the channels. The .dat files can be opened with the [LumiCycle](https://actimetrics.com/products/lumicycle/) Analysis program.

Experiment_number = 'CYxxx' #@param {type:"string"}
Experiment_title = '' #@param {type:"string"}
Date_experiment_started = '2026-01-01' #@param {type:"date"}

Detrending_Method = "Sinc Filter" #@param ["Sinc Filter", "Moving Average"]

# Sinc Filter Parameters (only used if 'Sinc Filter' is selected)
Sinc_Filter_Cutoff_Period_Hours = 48 # @param {type:"slider", min:1, max:240, step:1}
Sinc_Filter_Order = 101 # @param {type:"slider", min:1, max:361, step:2}

# For detrending by moving average, ste an window size
Window_size_for_trend_line_moving_average = 24 # @param {type:"slider", min:1, max:120, step:1}

Data_Plotting = "Plotting the channel 00 data last" #@param ["Plotting the channel 00 data first", "Plotting the channel 00 data last"]

# Plotting the data from the channel 0 or from the channel 1
if Data_Plotting == "Plotting the channel 00 data first":
	chlist = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29]
else:
	chlist = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,0]

# Subplot Mmatrix
Subplot_Matrix = "3-column by 10-row" #@param ["3-column by 10-row", "4-column by 8-row"]

if Subplot_Matrix == "3-column by 10-row":
	subp_raw = 10
	subp_col = 3
else:
	subp_raw = 8
	subp_col = 4

# Setting the X-axis major tick interval
Major_Ticks = "Every 24 hours" #@param ["Every 12 hours", "Every 24 hours", "Every 48 hours"]

major_ticks_map = {
	"Every 12 hours": 12,
	"Every 24 hours": 24,
	"Every 48 hours": 48
}
mjt = major_ticks_map.get(Major_Ticks)

# Setting the X-axis minor tick interval
Minor_Ticks = "Every 12 hours" #@param ["No minor ticks", "Every 2 hours", "Every 4 hours", "Every 6 hours", "Every 12 hours", "Every 24 hours"]

minor_ticks_map = {
	"No minor ticks": 0,
	"Every 2 hours": 2,
	"Every 4 hours": 4,
	"Every 6 hours": 6,
	"Every 12 hours": 12,
	"Every 24 hours": 24
}
mit = minor_ticks_map.get(Minor_Ticks)

# Determining whether to show minor ticks
if mit == 0 or mit >= mjt:
	miton = False
else:
	miton = True

# Determining whether to label peaks and troughs in detrended data plot
Label_peaks_and_troughs_in_detrended_data_plot = "Yes" #@param ["Yes", "No"]

if Label_peaks_and_troughs_in_detrended_data_plot == "Yes":
	sub2label = True
else:
	sub2label = False

# Daylength for double plot
Actogram_X_axis_scale = 24 # @param {type:"slider", min:12, max:48, step:0.1}
lod = Actogram_X_axis_scale

# Determining whether to label peaks and troughs in actogram
Label_peaks_and_troughs_in_actogram = "Yes" #@param ["Yes", "No"]

if Label_peaks_and_troughs_in_actogram == "Yes":
	sub3label = True
else:
	sub3label = False

### Deleting old data files
!rm -r -f *.xlsx *.pdf *.dat *.zip TRACES.* Traces.* traces.*

### Uploading TRACES.xxx file
uploaded = files.upload()
ttronfilename = next(iter(uploaded))

### Reading the uploaded TRACES file to a dataframe
print('\nReading the data...')

colnames = ["Hours","00","01","02","03","04","05","06","07","08","09","10","11","12","13","14","15","16","17","18","19","20","21","22","23","24","25","26","27","28","29"]

df = pd.read_csv(ttronfilename, header=None, sep ='\t', skiprows=3, skipfooter=1, index_col=False, names=colnames, engine='python')
df2= df.iloc[:,0:31]

print('\nRaw Data (first 5 rows & last 5 rows):')
display(df.head())
display(df.tail())

number_of_rows = len(df.index)
last_row_index = number_of_rows - 1
print('Number of rows: ' + str(number_of_rows))
print('First time points: ' + str(df.loc[0]['Hours']) + ' h')
print('Last time points: ' + str(df.loc[last_row_index]['Hours']) + ' h')
total_time = df.loc[last_row_index]['Hours'] - df.loc[0]['Hours']
total_time_in_days = total_time * (1/24)
print('Total time duration: ' + str(total_time) + ' h = ' + str(total_time_in_days) + ' days')
time_interval = total_time / last_row_index
excel2_sheet_name = 'INTVL = '+str('{:.15f}'.format(time_interval))+' h'
print('Average time interval: ' + str(time_interval) + ' h')
last_time_point = df.loc[last_row_index]['Hours']

####### Calculating moving averages and trend line #######
print('\nCalculating trend line and detrended data using ' + Detrending_Method + '...')

df_5PMA = df.rolling(window=5, center=True, min_periods=1).mean()
df_9PMA = df.rolling(window=9, center=True, min_periods=1).mean()

if Detrending_Method == "Moving Average":
    ###### Moving average time window for trend line
    tws = math.ceil(Window_size_for_trend_line_moving_average/time_interval)
    if tws%2 == 0: tws = tws + 1
    twss = int(tws)
    twst = time_interval * (twss - 1)
    print('Window size for trend line (Moving Average): ' + str(twss) + ' points (' + str('{:.6f}'.format(twst)) + ' h)')
    df_TL_MA = df.rolling(window=twss, center=True, min_periods=1).mean()
    # Fill NaN values at the boundaries for a complete trend line
    df_TL_MA = df_TL_MA.fillna(method='bfill').fillna(method='ffill')
    trendline_sheet_name = 'Trend line ('+str(twss)+'PMA)'

elif Detrending_Method == "Sinc Filter":
    # Sinc Filter Parameters Calculation
    sampling_rate = 1 / time_interval # samples per hour
    cutoff_frequency = 1 / Sinc_Filter_Cutoff_Period_Hours # cycles per hour
    nyquist_frequency = 0.5 * sampling_rate
    norm_cutoff = cutoff_frequency / nyquist_frequency
    filter_order = Sinc_Filter_Order

    # Ensure filter order is not too large for filtfilt
    max_filter_order_for_data = int(number_of_rows / 3) - 1
    if filter_order >= max_filter_order_for_data:
        print(f"Warning: Sinc Filter Order ({filter_order}) is too high for data length ({number_of_rows} points).")
        filter_order = max_filter_order_for_data
        if filter_order % 2 == 0: # Ensure filter order is odd
            filter_order -= 1 # Reduce by 1 if even
        if filter_order <= 0: # Ensure it's at least 1 if data is very short
            filter_order = 1
        print(f"Adjusting Sinc Filter Order to {filter_order} for stability with filtfilt.")

    if filter_order % 2 == 0: # Ensure filter order is odd for filtfilt stability
        filter_order += 1

    print(f"Sampling Rate: {sampling_rate:.2f} samples/hour")
    print(f"Cutoff Period for Sinc Filter: {Sinc_Filter_Cutoff_Period_Hours} hours")
    print(f"Cutoff Frequency: {cutoff_frequency:.4f} cycles/hour")
    print(f"Normalized Cutoff Frequency: {norm_cutoff:.4f}")
    print(f"Sinc Filter Order: {filter_order}")

    # Design the FIR filter (sinc-like filter)
    sinc_filter_coeffs = signal.firwin(filter_order, norm_cutoff, pass_zero='lowpass')

    # Create a new DataFrame for sinc-derived trend
    df_TL_MA = df.copy() # Use df_TL_MA for the trend output for consistency

    # Apply the filter to each channel to get the trend
    for k in range(0, 30, 1):
        channelnumber = str(k).zfill(2)
        if df[channelnumber].isnull().all(): # Skip if channel data is all NaN
            df_TL_MA[channelnumber] = np.nan
            continue

        # Handle NaNs: interpolate or fill before filtering.
        # Using linear interpolation for better results than mean fill.
        data_to_filter = df[channelnumber].interpolate(method='linear', limit_direction='both', axis=0)

        # Apply the filter using filtfilt for zero-phase distortion
        # Use try-except to catch potential issues with short data or filter order
        try:
            df_TL_MA[channelnumber] = signal.filtfilt(sinc_filter_coeffs, [1.0], data_to_filter)
        except ValueError as e:
            print(f"Warning: Could not apply sinc filter to channel {channelnumber}. Error: {e}")
            print("Consider reducing filter_order or increasing data length.")
            df_TL_MA[channelnumber] = np.nan # Set to NaN if filtering fails

    trendline_sheet_name = f'Trend line (Sinc Filter C={Sinc_Filter_Cutoff_Period_Hours}h O={filter_order})'

# Detrend the data by subtracting the trend line
dtdf = df - df_TL_MA
dtdf['Hours'] = df_TL_MA['Hours'] # Keep the 'Hours' column intact
dtdf_5PMA = dtdf.rolling(window=5, center=True, min_periods=1).mean()
dtdf_9PMA = dtdf.rolling(window=9, center=True, min_periods=1).mean()

### Generating an Excel file containing all the data
print('\nGenerating an Excel file...')
now = datetime.now(timezone.utc)
processed_dnt_str = now.strftime("%Y-%m-%d %H:%M:%S")
note_df = pd.DataFrame.from_dict(
	{
		'A': ['Experiment Number', 'Experiment Title', 'Experiment Start Date', 'TRACES File', '', 'Number of Time Points', 'Total Time Duration (Hours)', 'Average Time Interval (Hours)', 'Detrending Method', 'Trend Line Parameter', '', 'Data Processed Date and Time (UTC)'] ,
		'B': ['', '', '', '', '', '', '', '', '', '', '', ''],
		'C': ['', '', '', '', '', '', '', '', '', '', '', ''],
		'D': ['', '', '', '', '', '', '', '', '', '', '', ''],
		'E': [Experiment_number, Experiment_title, Date_experiment_started, ttronfilename, '', number_of_rows, total_time, time_interval, Detrending_Method, (f'{Window_size_for_trend_line_moving_average}h window' if Detrending_Method == 'Moving Average' else f'{Sinc_Filter_Cutoff_Period_Hours}h cutoff, {Sinc_Filter_Order} order'), '', processed_dnt_str]
	}
)

outputexcelfilename = Experiment_number+"_data.xlsx"
with pd.ExcelWriter(outputexcelfilename) as writer:
	note_df.to_excel(writer, sheet_name='Note', index=None, header=False)
	df.to_excel(writer, sheet_name='Raw Data')
	df_5PMA.to_excel(writer, sheet_name='5-point moving average (5PMA)')
	df_9PMA.to_excel(writer, sheet_name='9-point moving average (9PMA)')
	df_TL_MA.to_excel(writer, sheet_name=trendline_sheet_name)
	dtdf.to_excel(writer, sheet_name='Detrended Data')
	dtdf_5PMA.to_excel(writer, sheet_name='Detrended Data 5PMA')
	dtdf_9PMA.to_excel(writer, sheet_name='Detrended Data 9PMA')
	for k in range(0, 30, 1):
		channelnumber = str(k).zfill(2)
		chnum = 'Channel '+channelnumber
		dfx = pd.DataFrame()
		dfx['Hours'] = df['Hours']
		dfx['Raw_data'] = df[channelnumber]
		dfx['5PMA'] = df_5PMA[channelnumber]
		dfx['9PMA'] = df_9PMA[channelnumber]
		dfx['trend_line'] = df_TL_MA[channelnumber]
		dfx['detrended_data'] = dtdf[channelnumber]
		dfx['detrended_data_5PMA'] = dtdf_5PMA[channelnumber]
		dfx['detrended_data_9MPA'] = dtdf_9PMA[channelnumber]
		dfx.to_excel(writer, sheet_name=chnum)

	# Add a new sheet for detected peaks and troughs
	all_peaks_troughs_data = []
	for k in range(0, 30, 1):
		channelnumber = str(k).zfill(2)
		peaks_indices, troughs_indices = find_peaks_and_troughs(dtdf_9PMA[channelnumber], distance_in_hours=12, time_interval=time_interval)

		for peak_index in peaks_indices:
			all_peaks_troughs_data.append({
				'Channel': channelnumber,
				'Type': 'Peak',
				'Time (Hours)': dtdf_9PMA.loc[peak_index, 'Hours']
			})
		for trough_index in troughs_indices:
			all_peaks_troughs_data.append({
				'Channel': channelnumber,
				'Type': 'Trough',
				'Time (Hours)': dtdf_9PMA.loc[trough_index, 'Hours']
			})

	if all_peaks_troughs_data:
		df_peaks_troughs = pd.DataFrame(all_peaks_troughs_data)
		df_peaks_troughs.to_excel(writer, sheet_name='Peaks and Troughs', index=False)
	else:
		print("No peaks or troughs detected for any channel.")

print('\nExcel file: '+outputexcelfilename+'  has been generated.')

### Generating an Excel file conatining only the raw data
outputexcelfilename2 = Experiment_number+"_data_2.xlsx"
with pd.ExcelWriter(outputexcelfilename2) as writer:
	df2.to_excel(writer, sheet_name=excel2_sheet_name, index=None, header=True)

print('\nExcel file: '+outputexcelfilename2+'  has been generated.')

### Generating .dat files
print('\nGenerating a data file for each channel (.dat files)...')
df['Days'] = df['Hours'] / 24.000
for k in range(0, 30, 1):
	channelnumber = str(k).zfill(2)
	datfilename = channelnumber + '.dat'
	df.to_csv(datfilename, header=False, index=False, sep ='	', columns=['Days',channelnumber])

### Packing all the .dat files into a zip file
print('\nPacking .dat files into a zip file...')
zip_output_filename = Experiment_number + '_data.zip'
!zip -r {zip_output_filename} ./*.dat

print('\nPlotting...')
plot_output_pdf = Experiment_number + "_data_plots.pdf"
x = df['Hours']
x_scale_min = int(math.floor(min(x)*(1/24)))*24
x_scale_max = int(math.ceil(max(x)*(1/12)))*12+12
xtickslist = list(range(x_scale_min, x_scale_max, mjt))

pp = PdfPages(plot_output_pdf)
plt.rcParams.update({'figure.max_open_warning': 0})

fig = plt.figure(figsize=(11, 8.5))
fig.subplots_adjust(hspace=0.15)
fig.suptitle(Experiment_number, fontsize=12)

plt.rc('font', size=5)
plt.rc('axes', titlesize=4)
plt.rc('axes', labelsize=4)
plt.rc('xtick', labelsize=4)
plt.rc('ytick', labelsize=4)
plt.rc('legend', fontsize=3)
plt.rc('figure', titlesize=5)

subplotnumber = 1
for k in chlist:
		channelnumber = str(k).zfill(2)
		ax = plt.subplot(subp_raw, subp_col, subplotnumber)
		x = df['Hours']
		y = df[channelnumber]
		x_cma = df_TL_MA['Hours']
		y_cma = df_TL_MA[channelnumber]

		print('Plotting Channel '+channelnumber)
		chlabel = 'Ch # '+channelnumber
		ax.scatter(x,y,s=0.1,c='blue', label=chlabel)
		ax.plot(x_cma,y_cma,'-r', linewidth=0.5, label='trend line ('+Detrending_Method+')')
		ax.set_xlim(x_scale_min-6, x_scale_max)
		y_scale_max = int(max(y)*1.100)
		ax.set_ylim(0, y_scale_max)
		ax.set_xticks(xtickslist)
		if miton == True:
			ax.xaxis.set_minor_locator(MultipleLocator(mit))
		ax.grid(True, linewidth=0.5, color='lightgray', linestyle='--')
		ax.legend(loc='upper right', fontsize=4)
		subplotnumber += 1

fig.text(0.50, 0.06, 'Time (hours)', horizontalalignment='center', fontsize = 10)
fig.text(0.08, 0.50, 'Bioluminescence', horizontalalignment='center', verticalalignment='center', rotation='vertical', fontsize = 10)
pp.savefig(fig)

print('\nPlotting...')
fig = plt.figure(figsize=(11, 8.5))
fig.subplots_adjust(hspace=0.15)
fig.suptitle(Experiment_number+' - detrended data ('+Detrending_Method+')', fontsize=12)

subplotnumber = 1
for k in chlist:
		channelnumber = str(k).zfill(2)
		ax = plt.subplot(subp_raw, subp_col, subplotnumber)
		y = df[channelnumber]
		x_dt = dtdf['Hours']
		y_dt = dtdf[channelnumber]
		x_dt_cma = dtdf_5PMA['Hours']
		y_dt_cma = dtdf_5PMA[channelnumber]
		chlabel = 'Ch # '+channelnumber+' - detrend'
		print('Plotting Channel '+channelnumber+' (detrended)')
		ax.scatter(x_dt,y_dt,s=0.1,c='blue', label=chlabel)
		ax.plot(x_dt_cma,y_dt_cma,'-r', linewidth=0.5)
		ax.set_xlim(x_scale_min-6, x_scale_max)
		ax.set_xticks(xtickslist)
		if miton == True:
			ax.xaxis.set_minor_locator(MultipleLocator(mit))
		ax.grid(True, linewidth=0.5, color='lightgray', linestyle='--')
		ax.legend(loc='upper right', fontsize=4)
		subplotnumber += 1

fig.text(0.50, 0.06, 'Time (hours)', horizontalalignment='center', fontsize = 10)
fig.text(0.08, 0.50, 'Detrended Bioluminescence', horizontalalignment='center', verticalalignment='center', rotation='vertical', fontsize = 10)
pp.savefig(fig)

print('\nPlotting...')
x = df['Hours']
x_scale_min = int(math.floor(min(x)*(1/24)))*24
x_scale_max = int(math.ceil(max(x)*(1/12)))*12+12
xtickslist = list(range(x_scale_min, x_scale_max, mjt))

plt.rc('font', size=10)
plt.rc('axes', titlesize=10)
plt.rc('axes', labelsize=10)
plt.rc('xtick', labelsize=9)
plt.rc('ytick', labelsize=9)
plt.rc('legend', fontsize=6)
plt.rc('figure', titlesize=12)

for k in chlist:
	channelnumber = str(k).zfill(2)
	fig = plt.figure(figsize=(8.5, 11)) # Increased figure height for 3 subplots
	fig.suptitle(Experiment_number+'   Ch # '+channelnumber, fontsize=12)

	# Subplot 1: Raw data with trend line
	ax1 = plt.subplot(3, 1, 1) # Changed to 3x1 subplot layout
	x = df['Hours']
	y = df[channelnumber]
	x_cma = df_TL_MA['Hours']
	y_cma = df_TL_MA[channelnumber]
	print('Plotting Channel '+channelnumber)
	ax1.scatter(x,y,s=3.0,c='violet', label='Bioluminescence')
	if Detrending_Method == "Moving Average":
		ax1.plot(x_cma,y_cma,'-r', linewidth=1.0, label='Trend line (' + str(twss) +'-point moving average, centered)')
	else: # Sinc Filter
		ax1.plot(x_cma,y_cma,'-r', linewidth=1.0, label=f'Trend line (Sinc Filter C={Sinc_Filter_Cutoff_Period_Hours}h O={filter_order})')
	ax1.set_xlim(x_scale_min, x_scale_max)
	y_scale_max = int(max(y)*1.100)
	ax1.set_ylim(0, y_scale_max)
	ax1.set_xticks(xtickslist)
	if miton == True:
		ax1.xaxis.set_minor_locator(MultipleLocator(mit))
	ax1.set_xlabel('Hours', fontsize=10)
	ax1.set_ylabel('Bioluminescence', fontsize=10)
	ax1.grid(True, linewidth=0.5, color='lightgray', linestyle='--')
	ax1.legend(loc='upper right', fontsize=5)

	# Subplot 2: Detrended data with smoothed line and peaks/troughs
	ax2 = plt.subplot(3, 1, 2) # Changed to 3x1 subplot layout
	x_dt = dtdf['Hours']
	y_dt = dtdf[channelnumber]
	x_dt_cma = dtdf_9PMA['Hours']
	y_dt_cma = dtdf_9PMA[channelnumber]
	print('Plotting Channel '+channelnumber+' (detrended)')
	ax2.scatter(x_dt,y_dt,s=3.0,c='violet', label='Detrended bioluminescence')
	ax2.plot(x_dt_cma,y_dt_cma,'-b', linewidth=1.0, label='Smoothed line (9-point moving average, centered)')

	# Detect and plot peaks and troughs on the detrended data
	peaks_indices, troughs_indices = find_peaks_and_troughs(dtdf_9PMA[channelnumber], distance_in_hours=12, time_interval=time_interval)

	if len(peaks_indices) > 0:
		ax2.scatter(dtdf_9PMA.loc[peaks_indices, 'Hours'], dtdf_9PMA.loc[peaks_indices, channelnumber],
		           marker='o', s=30, color='red', label='Peaks')
		for idx in peaks_indices:
			peak_time = dtdf_9PMA.loc[idx, 'Hours']
			peak_value = dtdf_9PMA.loc[idx, channelnumber]
			if sub2label == True:
				ax2.text(peak_time, peak_value + (ax2.get_ylim()[1] - ax2.get_ylim()[0]) * 0.05, f'{peak_time:.2f} h', fontsize=5, color='red', ha='center', va='bottom')
	if len(troughs_indices) > 0:
		ax2.scatter(dtdf_9PMA.loc[troughs_indices, 'Hours'], dtdf_9PMA.loc[troughs_indices, channelnumber],
		           marker='o', s=30, color='blue', label='Troughs')
		for idx in troughs_indices:
			trough_time = dtdf_9PMA.loc[idx, 'Hours']
			trough_value = dtdf_9PMA.loc[idx, channelnumber]
			if sub2label == True:
				ax2.text(trough_time, trough_value - (ax2.get_ylim()[1] - ax2.get_ylim()[0]) * 0.05, f'{trough_time:.2f} h', fontsize=5, color='blue', ha='center', va='top')

	ax2.set_xlim(x_scale_min, x_scale_max)
	ax2.set_xticks(xtickslist)
	if miton == True:
		ax2.xaxis.set_minor_locator(MultipleLocator(mit))
	ax2.set_xlabel('Hours', fontsize=10)
	ax2.set_ylabel('Detrended bioluminescence', fontsize=10)
	ax2.grid(True, linewidth=0.5, color='lightgray', linestyle='--')
	ax2.legend(loc='upper right', fontsize=5)

	# Subplot 3: Actogram of peaks and troughs
	ax3 = plt.subplot(3, 1, 3) # New subplot for actogram

	# Extract peak and trough hours for actogram
	peak_hours_actogram = dtdf_9PMA.loc[peaks_indices, 'Hours'].values # Use dtdf_9PMA for consistency with detection
	trough_hours_actogram = dtdf_9PMA.loc[troughs_indices, 'Hours'].values # Use dtdf_9PMA for consistency with detection

	peak_actogram_data = []
	for h in peak_hours_actogram:
		day_num = h // lod
		time_in_day = h - day_num * lod
		peak_actogram_data.append((time_in_day, day_num, h)) # Store original h for label
		if day_num > 0:
			peak_actogram_data.append((time_in_day + lod, day_num -1, h )) # Double plot

	trough_actogram_data = []
	for h in trough_hours_actogram:
		day_num = h // lod
		time_in_day = h - day_num * lod
		trough_actogram_data.append((time_in_day, day_num, h)) # Store original h for label
		if day_num > 0:
			trough_actogram_data.append((time_in_day + lod, day_num - 1, h)) # Double plot

	# Plot peaks on actogram
	if peak_actogram_data:
		peak_x, peak_y, peak_original_h = zip(*peak_actogram_data) # Unpack original hours too
		ax3.scatter(peak_x, peak_y, marker='o', s=20, color='red', label='Peaks')
		if sub3label == True:
			for i in range(len(peak_x)):
				ax3.text(peak_x[i] + 0.5, peak_y[i], f'{peak_original_h[i]:.2f}', fontsize=5, color='red')

	# Plot troughs on actogram
	if trough_actogram_data:
		trough_x, trough_y, trough_original_h = zip(*trough_actogram_data) # Unpack original hours too
		ax3.scatter(trough_x, trough_y, marker='o', s=20, color='blue', label='Troughs')
		if sub3label == True:
			for i in range(len(trough_x)):
				ax3.text(trough_x[i] +0.5, trough_y[i], f'{trough_original_h[i]:.2f}', fontsize=5, color='blue')

	print('Plotting Channel '+channelnumber+' (actogram)')

	if sub3label == True:
		actogram_x_min = -0.085*lod
		actogram_x_max = 2.085*lod
		ax3.set_xlim(actogram_x_min, actogram_x_max)
	else:
		 ax3.set_xlim(0, lod*2) # Actogram spans 0-48 hours for double plot

	if lod == 24:
		ax3.set_xticks([0, 6, 12, 18, 24, 30, 36, 42, 48])
		ax3.set_xticklabels(['0', '6', '12', '18', '24', '30', '36', '42', '48'])
		ax3.set_title('Peaks and Troughs', fontsize=10)
		ax3.set_ylabel('Days', fontsize=10)
	else:
		ax3.set_xticks([0, (lod/4)*1, (lod/4)*2, (lod/4)*3, (lod/4)*4, (lod/4)*5, (lod/4)*6, (lod/4)*7, (lod/4)*8])
		ax3.set_xticklabels([str((lod/4)*0), str((lod/4)*1), str((lod/4)*2), str((lod/4)*3), str((lod/4)*4), str((lod/4)*5), str((lod/4)*6), str((lod/4)*7), str((lod/4)*8)])
		ax3.set_title('Peaks and Troughs (scaled x-axis: T = '+str(lod)+' hours, T'+str(lod)+')', fontsize=10)
		ax3.set_ylabel('Days (T'+str(lod)+')', fontsize=12)

	ltpd = last_time_point // lod
	ax3.set_ylim(-ltpd*0.05, ltpd*1.05)

	ax3.yaxis.set_major_locator(MultipleLocator(base=1, offset=0))
	ax3.set_xlabel('Time (Hours)', fontsize=10)
	ax3.grid(True, linestyle='--', alpha=0.7)
	ax3.legend(loc='upper right', fontsize=6)
	ax3.invert_yaxis() # Invert y-axis to show Day 0 at the top

	plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
	pp.savefig(fig)

pp.close()

files.download(outputexcelfilename)
files.download(outputexcelfilename2)
files.download(plot_output_pdf)
files.download(zip_output_filename)
